# DECOMP — interactive exploration

V1-vs-cerebellum movement signal decomposition on the IBL Brain-Wide Map.

This notebook walks through every stage of the pipeline implemented in `src/decomp/`. It is
complementary to `run_all.py` (the artifact-producing entry point); use this notebook for
diagnostics, parameter sweeps, and per-session sanity checks.

All design decisions and the supporting deep-research are logged at
`scratch/2026-05-04-v1-cb-movement-decomposition/`.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from one.api import ONE
from brainwidemap import bwm_query

from decomp.pipeline.stage_data import load_and_bin, select_sessions
from decomp.pipeline.stage_glm import run_session_glm
from decomp.pipeline.stage_svca import run_session_svca
from decomp.pipeline.stage_cca import run_session_cca
from decomp.viz.figures import (
    fig01_glm_dr2_per_region, fig02_svca_reliability,
    fig03_cca_canonical_correlations, fig04_pcca_vs_cca,
)

CACHE = Path('data/cache')
OUT = Path('outputs')

## 1. Session selection (Gate-1 fallback ladder)

In [2]:
one = ONE(base_url='https://openalyx.internationalbrainlab.org', password='international', silent=True)
plan = select_sessions(one, target_n=3, freeze='2023_12_bwm_release', cache_dir=CACHE)
print(plan.strategy_note)
plan.coverage.head()

Loading bwm_query results from fixtures/2023_12_bwm_release.csv


(S3) /Users/kevinb/Downloads/ONE/openalyx.internationalbrainlab.org/bwm_tables/trials.pqt: 100%|██████████| 24.2M/24.2M [00:01<00:00, 21.1MB/s]
(S3) /Users/kevinb/Downloads/ONE/openalyx.internationalbrainlab.org/bwm_tables/clusters.pqt: 100%|██████████| 91.6M/91.6M [00:01<00:00, 50.2MB/s]


d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01
V1+CB pair: 3 sessions at min_units=5 | per-region pool: VISp=3, CB=3, MO=3, CA1=3 sessions at min_units=10


,eid,VISp_units,VISp_ok,CB_units,CB_ok,MO_units,MO_ok,CA1_units,CA1_ok,all_ok
0,004d8fd5-41e7-4f1b-a45b-0d4ad76fe446,0,False,44,True,0,False,0,False,False
1,02fbb6da-3034-47d6-a61b-7d06c796a830,0,False,0,False,0,False,0,False,False
2,03063955-2523-47bd-ae57-f7489dd40f15,0,False,0,False,38,True,0,False,False
3,032452e9-1886-449d-9c13-0f192572e19f,0,False,0,False,0,False,0,False,False
4,032ffcdf-7692-40b3-b9ff-8def1fc18b2e,12,True,0,False,0,False,0,False,False


## 2. Load + bin one session

In [3]:
bwm = bwm_query(one)
eid_to_pids = bwm.groupby('eid')['pid'].apply(list).to_dict()
eid = plan.eids[0]
pids = eid_to_pids[eid]
sd, binned = load_and_bin(one, eid, pids, rois=plan.rois_used, cache_dir=CACHE)
print(f"{eid}: T={len(binned.bin_centers)} bins, ROIs={list(binned.spikes_by_roi)}")
for roi, mat in binned.spikes_by_roi.items():
    print(f"  {roi}: {mat.shape[0]} units")

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
Downloading: /Users/kevinb/Downloads/ONE/openalyx.internationalbrainlab.org/histology/ATLAS/Needles/Allen/average_template_25.nrrd Bytes: 32998960


100%|██████████| 31.470260620117188/31.470260620117188 [00:00<00:00, 62.41it/s]


Downloading: /Users/kevinb/Downloads/ONE/openalyx.internationalbrainlab.org/histology/ATLAS/Needles/Allen/annotation_25.nrrd Bytes: 4035363


100%|██████████| 3.848422050476074/3.848422050476074 [00:00<00:00, 23.67it/s]
(S3) /Users/kevinb/Downloads/ONE/openalyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-01577/2020-10-28/001/alf/probe01/pykilosort/#2024-05-06#/passingSpikes.table.pqt: 100%|██████████| 388M/388M [00:06<00:00, 56.5MB/s] 
(S3) /Users/kevinb/Downloads/ONE/openalyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-01577/2020-10-28/001/alf/probe01/pykilosort/#2024-05-06#/clusters.channels.npy: 100%|██████████| 9.42k/9.42k [00:00<00:00, 45.3kB/s]
(S3) /Users/kevinb/Downloads/ONE/openalyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-01577/2020-10-28/001/alf/probe01/pykilosort/#2024-05-06#/clusters.depths.npy: 100%|██████████| 4.77k/4.77k [00:00<00:00, 20.6kB/s]
(S3) /Users/kevinb/Downloads/ONE/openalyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-01577/2020-10-28/001/alf/probe01/pykilosort/#2024-05-06#/clusters.metrics.pqt: 100%|██████████| 161k/161k [00:00<00:00, 367kB/s]
(S3) /Users/kevinb/Downloads/ON

09b2c4d1-058d-4c84-9fd4-97530f85baf6: T=262916 bins, ROIs=['CB', 'VISp', 'CA1']
  CB: 9 units
  VISp: 51 units
  CA1: 20 units


## 3. Per-region GLM ΔR² (sanity check vs IBL BWM / Wang 2026)

In [4]:
glm_df = run_session_glm(sd, binned, rois=plan.rois_used, cache_dir=CACHE)
glm_df.groupby('region')[['dR2_movement','dR2_stim','dR2_choice']].median()

GLM 09b2c4d1 CA1 (20 units): 100%|██████████| 1/1 [00:03<00:00,  3.82s/it]


,dR2_movement,dR2_stim,dR2_choice
region,,,
CA1,0.000385,0.000326,0.000067
CB,0.000231,0.009456,0.000269
VISp,0.000269,0.000305,0.000050


In [5]:
fig01_glm_dr2_per_region(glm_df, OUT)

## 4. Within-region SVCA

In [6]:
svca = run_session_svca(binned, cache_dir=CACHE)
for roi, res in svca.items():
    print(f"{roi}: k_reliable={res.k_reliable}, top reliability={res.reliability[:5]}")
fig02_svca_reliability(svca, OUT)

CB: k_reliable=1, top reliability=[0.22825602 0.25096142 0.12845317]
VISp: k_reliable=1, top reliability=[0.40602615 0.29594347 0.15629895 0.17544733 0.10005143]
CA1: k_reliable=1, top reliability=[0.11475956 0.33371061 0.17809665 0.04965739 0.05460589]


## 5. Pairwise CCA + pCCA — the answer

In [7]:
cca_df = run_session_cca(binned, svca, cache_dir=CACHE,
                          n_components=8, n_surrogates=100)
fig03_cca_canonical_correlations(cca_df, OUT)
fig04_pcca_vs_cca(cca_df, OUT)
cca_df.groupby(['pair_a','pair_b'])[['rho_cca','rho_pcca']].sum()

rho_cca  rho_pcca
pair_a pair_b                    
CB     CA1     0.350945  0.310113
       VISp    0.487735  0.427495
VISp   CA1     0.574392  0.511181